In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [2]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

# 1. Dzielimy dane na treningowe (80%) i testowe (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# 2. Tworzymy i trenujemy model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Oceniamy na danych testowych
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# 4. Zapisujemy model do pliku
pickle.dump(model, open('fraud_model.pkl', 'wb'))
print("Zapisano fraud_model.pkl")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Zapisano fraud_model.pkl


In [8]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score(tx: Transaction):
    features = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    proba = model.predict_proba(features)[0][1]
    return {
        "is_fraud": bool(proba >= 0.5),
        "fraud_probability": round(float(proba), 4)
    }

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

Overwriting fraud_api.py


In [4]:
import requests

# Test 1 — normalna transakcja
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

# Test 2 — podejrzana transakcja
r = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Podejrzana:", r.json())

Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}


In [5]:
%%file producer.py
from kafka import KafkaProducer
from datetime import datetime
import json, time, random

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

print("Producer wystartował. Wysyłam transakcje na temat 'transactions'... (Ctrl+C aby przerwać)")

i = 0
while True:
    # co jakiś czas wygeneruj podejrzaną transakcję
    if random.random() < 0.2:
        tx = {
            "amount": round(random.uniform(2000, 9000), 2),
            "is_electronics": 1,
            "tx_per_minute": random.randint(8, 15)
        }
    else:
        tx = {
            "amount": round(random.uniform(5, 500), 2),
            "is_electronics": random.randint(0, 1),
            "tx_per_minute": random.randint(1, 5)
        }

    tx["timestamp"] = datetime.now().isoformat()

    producer.send('transactions', tx)
    producer.flush()
    i += 1
    print(f"[{i}] Wysłano: {tx}")
    time.sleep(2)

Writing producer.py


In [6]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

print("ML consumer wystartował. Czekam na transakcje...")

for message in consumer:
    tx = message.value

    # 1. Wyciągnij cechy, których oczekuje API
    features = {
        "amount": tx["amount"],
        "is_electronics": tx.get("is_electronics", 0),
        "tx_per_minute": tx.get("tx_per_minute", 5)
    }

    # 2. Zapytaj API o ocenę
    response = requests.post(API_URL, json=features)
    result = response.json()

    # 3. Jeśli fraud — alert na temat 'alerts' + wypisz
    if result["is_fraud"]:
        alert = {
            "transaction": tx,
            "fraud_probability": result["fraud_probability"],
            "detected_at": datetime.now().isoformat()
        }
        alert_producer.send('alerts', alert)
        alert_producer.flush()
        print(f"🚨 ALERT! fraud_prob={result['fraud_probability']:.2f} | {tx}")
    else:
        print(f"   ok    fraud_prob={result['fraud_probability']:.2f} | amount={tx['amount']}")

Writing ml_consumer.py


In [7]:
from sklearn.metrics import classification_report

# --- Scoring REGUŁOWY (w stylu Ćw. 1) ---
# Reguła: oznacz jako fraud, jeśli kwota duża ORAZ (elektronika LUB dużo tx/min)
def rule_based(row):
    if row['amount'] > 2000 and (row['is_electronics'] == 1 or row['tx_per_minute'] >= 6):
        return 1
    return 0

y_pred_rule = X_test.apply(rule_based, axis=1)

# --- Scoring ML (nasz RandomForest) ---
y_pred_ml = model.predict(X_test)

print("=== REGUŁA ===")
print(classification_report(y_test, y_pred_rule, digits=2))
print("=== ML (RandomForest) ===")
print(classification_report(y_test, y_pred_ml, digits=2))

=== REGUŁA ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      0.90      0.95        20

    accuracy                           1.00       420
   macro avg       1.00      0.95      0.97       420
weighted avg       1.00      1.00      1.00       420

=== ML (RandomForest) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420



In [9]:
import requests
r = requests.get("http://localhost:8001/health")
print(r.json())

{'status': 'ok', 'model_loaded': True}
